# GRU and CatBoost weather blend

Reviewed by Tomislav Novosel and the team on 22 October 2026. Prepared with AI assistance. See [AI use](../docs/AI_USE.md).

## Why this model

GRU reads counts in time order. CatBoost learns additional relationships. Their 75%/25% blend has the lowest recorded MAE, 13,263. Uncertainty checks still include no improvement over XGBoost.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/review_models.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the project.')
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'mushrooms'), str(ROOT / 'citibike'), str(ROOT)]
import pandas as pd
from IPython.display import display
from src.review_models import load_model, bike_example, predict_bike
metadata = json.loads((ROOT / 'reports/research/citibike_weather_gru_metadata.json').read_text())
configuration = metadata['configuration']
display(pd.DataFrame([{'family': r['family'], 'settings': r['configuration']} for r in configuration['members']]))
print('Weights:', configuration['weights'])
display(predict_bike(bike_example('weather_gru')))

,family,settings
0,sequence_history_weather_gru,"{'architecture': 'gru', 'hidden': 24, 'head': ..."
1,history_weather_catboost,"{'feature_set': 'weather_full', 'target': 'del..."


Weights: [0.75, 0.25]


{'model': '75% GRU / 25% CatBoost',
 'prediction_date': '2023-06-30',
 'predicted_rides': 122094,
 'unrounded_prediction': 122094.28098957065,
 'note': 'One-day forecast using completed prior counts. Historical example; no live feed is connected.'}

## Optional exact training recipe

Saved prediction works on CPU. This training implementation needs CUDA. It chooses epochs on an inner training split, then refits the outer training rows. Keep summer and final-test targets out.

In [2]:
RUN_GPU_REPRODUCTION = False
if RUN_GPU_REPRODUCTION:
    from datetime import datetime, timedelta, timezone
    import os, torch, joblib
    from expanded_search import load_extended_weather_features
    from selection import estimator_for_record
    if not torch.cuda.is_available():
        raise RuntimeError('Use the CPU alternative or a verified CUDA machine.')
    os.environ['CLOUDAI_RESEARCH_DEADLINE_UTC'] = (datetime.now(timezone.utc) + timedelta(hours=2)).isoformat()
    frozen = json.loads((ROOT / 'reports/research/frozen_finalists.json').read_text())
    entry = next(r for r in frozen['finalists'] if r['run_id'] == 'blend_expanded_history_and_forecasts')
    X, y, _ = load_extended_weather_features()
    model = estimator_for_record(entry).fit(X, y)
    OUT = ROOT / 'reports/reproduction/citibike'
    OUT.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, OUT / 'weather_gru.joblib', compress=3)
else:
    print('GPU reproduction not run. The recorded research fit is preserved.')

GPU reproduction not run. The recorded research fit is preserved.
